# NZDep2023 Socioeconomic Context Experiment

## Objective

This experiment investigates whether area-level socioeconomic deprivation
provides additional predictive information for the severity classification
of recorded New Zealand road crashes.

The prediction target remains crash severity:

- Non-Injury Crash
- Minor Crash
- Serious Crash
- Fatal Crash

This experiment does not predict crash occurrence or crash risk.

NZDep2023 is treated as an area-level contextual feature. Any observed
association or model contribution will not be interpreted as causal.

In [1]:
import pandas as pd
from pathlib import Path


# 1. Find project root automatically

CURRENT_FOLDER = Path.cwd()

PROJECT_ROOT = None

for folder in [CURRENT_FOLDER, *CURRENT_FOLDER.parents]:

    cas_candidate = (
        folder
        / "data"
        / "CAS_Data_public.csv"
    )

    nzdep_candidate = (
        folder
        / "data"
        / "nzdep2023"
        / "NZDep2023_MB2023.txt"
    )

    if (
        cas_candidate.exists()
        and nzdep_candidate.exists()
    ):
        PROJECT_ROOT = folder
        break


if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root containing both "
        "data/CAS_Data_public.csv and "
        "data/nzdep2023/NZDep2023_MB2023.txt."
    )


CAS_PATH = (
    PROJECT_ROOT
    / "data"
    / "CAS_Data_public.csv"
)

NZDEP_PATH = (
    PROJECT_ROOT
    / "data"
    / "nzdep2023"
    / "NZDep2023_MB2023.txt"
)


print(
    "Project root:",
    PROJECT_ROOT
)


# 2. Load datasets

cas = pd.read_csv(
    CAS_PATH,
    low_memory=False
)

nzdep = pd.read_csv(
    NZDEP_PATH,
    sep="\t",
    dtype={
        "MB2023_code": "string",
        "SA12023_code": "string"
    }
)


print(
    "CAS shape:",
    cas.shape
)

print(
    "NZDep2023 shape:",
    nzdep.shape
)

print(
    "\nNZDep2023 columns:"
)

print(
    nzdep.columns.tolist()
)


# 3. Inspect CAS meshblock ID

print(
    "\nCAS meshblockId summary:"
)

print(
    cas["meshblockId"].describe()
)


print(
    "\nFirst CAS meshblock IDs:"
)

print(
    cas["meshblockId"]
    .dropna()
    .head(10)
    .tolist()
)


print(
    "\nFirst NZDep meshblock IDs:"
)

print(
    nzdep["MB2023_code"]
    .dropna()
    .head(10)
    .tolist()
)


# 4. Normalise IDs for compatibility test

cas["meshblock_key"] = (
    pd.to_numeric(
        cas["meshblockId"],
        errors="coerce"
    )
    .astype("Int64")
)

nzdep["meshblock_key"] = (
    pd.to_numeric(
        nzdep["MB2023_code"],
        errors="coerce"
    )
    .astype("Int64")
)


# 5. Test overlap without merging

valid_cas = (
    cas[
        "meshblock_key"
    ].notna()
)

nzdep_keys = set(
    nzdep[
        "meshblock_key"
    ]
    .dropna()
)

matched = (
    cas.loc[
        valid_cas,
        "meshblock_key"
    ]
    .isin(
        nzdep_keys
    )
)


print(
    "\nCAS crashes with non-missing meshblockId:",
    valid_cas.sum()
)

print(
    "Meshblock IDs matching NZDep2023:",
    matched.sum()
)

print(
    "Match rate among non-missing CAS meshblocks:",
    round(
        matched.mean() * 100,
        2
    ),
    "%"
)

print(
    "\nUnique CAS meshblocks:",
    cas[
        "meshblock_key"
    ].nunique()
)

print(
    "Unique NZDep2023 meshblocks:",
    nzdep[
        "meshblock_key"
    ].nunique()
)

Project root: C:\Users\ASUS\Desktop\Project_Roadcrash
CAS shape: (34866, 72)
NZDep2023 shape: (56382, 4)

NZDep2023 columns:
['MB2023_code', 'NZDep2023', 'NZDep2023_Score', 'SA12023_code']

CAS meshblockId summary:
count    3.472800e+04
mean     1.393114e+06
std      9.025352e+05
min      1.000000e+02
25%      6.809518e+05
50%      1.207203e+06
75%      2.168575e+06
max      3.205003e+06
Name: meshblockId, dtype: float64

First CAS meshblock IDs:
[639100.0, 1125400.0, 1274300.0, 2078900.0, 111700.0, 780403.0, 1278104.0, 175114.0, 1329300.0, 2375700.0]

First NZDep meshblock IDs:
['100', '200', '300', '400', '502', '600', '700', '800', '900', '1000']

CAS crashes with non-missing meshblockId: 34728
Meshblock IDs matching NZDep2023: 24778
Match rate among non-missing CAS meshblocks: 71.35 %

Unique CAS meshblocks: 15580
Unique NZDep2023 meshblocks: 56382


In [2]:
# Download official Stats NZ 2023 meshblock polygons
# Uses POST for feature downloads

import geopandas as gpd
import pandas as pd
import requests
from pathlib import Path
import time


# 1. Find project root automatically

CURRENT_FOLDER = Path.cwd()

PROJECT_ROOT = None

for folder in [CURRENT_FOLDER, *CURRENT_FOLDER.parents]:
    if (
        folder
        / "data"
        / "nzdep2023"
    ).exists():
        PROJECT_ROOT = folder
        break


if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root containing "
        "data/nzdep2023."
    )


# 2. Service and save paths

MB_SERVICE = (
    "https://services2.arcgis.com/vKb0s8tBIA3bdocZ/"
    "ArcGIS/rest/services/Meshblock_2023/FeatureServer/0/query"
)

SAVE_DIR = (
    PROJECT_ROOT
    / "data"
    / "nzdep2023"
)

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MB_SAVE_PATH = (
    SAVE_DIR
    / "Meshblock_2023_2193.gpkg"
)


print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "Meshblock save path:",
    MB_SAVE_PATH
)


# 3. Get all object IDs

params_ids = {
    "where": "1=1",
    "returnIdsOnly": "true",
    "f": "json"
}

response = requests.get(
    MB_SERVICE,
    params=params_ids,
    timeout=60
)

response.raise_for_status()

object_ids = response.json()[
    "objectIds"
]

print(
    "Total meshblock polygon records:",
    len(object_ids)
)


# 4. Download features in chunks using POST

chunk_size = 500

gdf_parts = []

session = requests.Session()


for start in range(
    0,
    len(object_ids),
    chunk_size
):

    chunk_ids = object_ids[
        start:start + chunk_size
    ]

    params = {
        "objectIds": ",".join(
            map(
                str,
                chunk_ids
            )
        ),
        "outFields": (
            "OBJECTID,"
            "MB2023_V1_00,"
            "LANDWATER,"
            "LANDWATER_NAME,"
            "LAND_AREA_SQ_KM"
        ),
        "returnGeometry": "true",
        "outSR": "2193",
        "f": "geojson"
    }

    r = session.post(
        MB_SERVICE,
        data=params,
        timeout=120
    )

    if r.status_code != 200:

        print(
            "\nFailed chunk starting at:",
            start
        )

        print(
            "HTTP status:",
            r.status_code
        )

        print(
            "Server response:",
            r.text[:1000]
        )

        raise RuntimeError(
            "Meshblock download failed."
        )


    geojson = r.json()


    if "features" not in geojson:

        print(
            "\nUnexpected response at chunk:",
            start
        )

        print(
            geojson
        )

        raise RuntimeError(
            "No features returned."
        )


    part = gpd.GeoDataFrame.from_features(
        geojson["features"],
        crs="EPSG:2193"
    )

    gdf_parts.append(
        part
    )


    downloaded = min(
        start + chunk_size,
        len(object_ids)
    )


    print(
        f"Downloaded {downloaded:,} "
        f"/ {len(object_ids):,}"
    )


    time.sleep(
        0.1
    )


# 5. Combine all chunks

meshblocks_2023 = pd.concat(
    gdf_parts,
    ignore_index=True
)

meshblocks_2023 = gpd.GeoDataFrame(
    meshblocks_2023,
    geometry="geometry",
    crs="EPSG:2193"
)


# 6. Validation

print(
    "\nMeshblock GeoDataFrame shape:"
)

print(
    meshblocks_2023.shape
)


print(
    "\nCRS:"
)

print(
    meshblocks_2023.crs
)


print(
    "\nGeometry types:"
)

print(
    meshblocks_2023.geometry
    .geom_type
    .value_counts(
        dropna=False
    )
)


print(
    "\nMissing geometry:"
)

print(
    meshblocks_2023.geometry
    .isna()
    .sum()
)


print(
    "\nUnique meshblock codes:"
)

print(
    meshblocks_2023[
        "MB2023_V1_00"
    ].nunique()
)


# 7. Save locally

meshblocks_2023.to_file(
    MB_SAVE_PATH,
    layer="meshblock_2023",
    driver="GPKG"
)


print(
    "\nSaved successfully:"
)

print(
    MB_SAVE_PATH
)

Project root: C:\Users\ASUS\Desktop\Project_Roadcrash
Meshblock save path: C:\Users\ASUS\Desktop\Project_Roadcrash\data\nzdep2023\Meshblock_2023_2193.gpkg
Total meshblock polygon records: 57539
Downloaded 500 / 57,539
Downloaded 1,000 / 57,539
Downloaded 1,500 / 57,539
Downloaded 2,000 / 57,539
Downloaded 2,500 / 57,539
Downloaded 3,000 / 57,539
Downloaded 3,500 / 57,539
Downloaded 4,000 / 57,539
Downloaded 4,500 / 57,539
Downloaded 5,000 / 57,539
Downloaded 5,500 / 57,539
Downloaded 6,000 / 57,539
Downloaded 6,500 / 57,539
Downloaded 7,000 / 57,539
Downloaded 7,500 / 57,539
Downloaded 8,000 / 57,539
Downloaded 8,500 / 57,539
Downloaded 9,000 / 57,539
Downloaded 9,500 / 57,539
Downloaded 10,000 / 57,539
Downloaded 10,500 / 57,539
Downloaded 11,000 / 57,539
Downloaded 11,500 / 57,539
Downloaded 12,000 / 57,539
Downloaded 12,500 / 57,539
Downloaded 13,000 / 57,539
Downloaded 13,500 / 57,539
Downloaded 14,000 / 57,539
Downloaded 14,500 / 57,539
Downloaded 15,000 / 57,539
Downloaded 15,500

In [3]:
# Spatially assign CAS crashes to 2023 meshblocks

import geopandas as gpd
import pandas as pd
import numpy as np


# 1. Prepare CAS coordinates

cas_spatial = cas.copy()

cas_spatial["X"] = pd.to_numeric(
    cas_spatial["X"],
    errors="coerce"
)

cas_spatial["Y"] = pd.to_numeric(
    cas_spatial["Y"],
    errors="coerce"
)

# Stable row identifier
cas_spatial["cas_row_id"] = np.arange(
    len(cas_spatial)
)


print(
    "Total CAS crashes:",
    len(cas_spatial)
)

print(
    "Missing X coordinates:",
    cas_spatial["X"].isna().sum()
)

print(
    "Missing Y coordinates:",
    cas_spatial["Y"].isna().sum()
)


# 2. Create crash point geometry
# NZTM2000 = EPSG:2193

valid_xy = (
    cas_spatial["X"].notna()
    & cas_spatial["Y"].notna()
)

crash_gdf_2023 = gpd.GeoDataFrame(
    cas_spatial.loc[
        valid_xy
    ].copy(),
    geometry=gpd.points_from_xy(
        cas_spatial.loc[
            valid_xy,
            "X"
        ],
        cas_spatial.loc[
            valid_xy,
            "Y"
        ]
    ),
    crs="EPSG:2193"
)


print(
    "\nCrash points created:",
    len(crash_gdf_2023)
)

print(
    "Crash CRS:",
    crash_gdf_2023.crs
)


# 3. Prepare valid meshblock polygons

mb_valid = (
    meshblocks_2023[
        meshblocks_2023.geometry.notna()
    ]
    [
        [
            "MB2023_V1_00",
            "LANDWATER",
            "LANDWATER_NAME",
            "LAND_AREA_SQ_KM",
            "geometry"
        ]
    ]
    .copy()
)

mb_valid["MB2023_V1_00"] = (
    mb_valid[
        "MB2023_V1_00"
    ]
    .astype("string")
)


print(
    "\nUsable meshblock polygons:",
    len(mb_valid)
)


# 4. Spatial join
# A crash receives the code of the 2023 meshblock
# polygon containing its coordinates.

crash_mb = gpd.sjoin(
    crash_gdf_2023,
    mb_valid,
    how="left",
    predicate="within"
)


# 5. Check for duplicate crash matches

match_counts = (
    crash_mb[
        "cas_row_id"
    ]
    .value_counts()
)

duplicate_matches = (
    match_counts > 1
).sum()


print(
    "\nCrash points matched to multiple meshblocks:",
    duplicate_matches
)


# 6. Spatial match coverage

matched_mask = (
    crash_mb[
        "MB2023_V1_00"
    ]
    .notna()
)

n_matched = (
    matched_mask.sum()
)

n_unmatched = (
    len(crash_gdf_2023)
    - n_matched
)

match_rate = (
    n_matched
    / len(crash_gdf_2023)
    * 100
)


print(
    "\nSpatial Match Results"
)

print(
    "Crash points:",
    len(crash_gdf_2023)
)

print(
    "Matched to 2023 meshblock:",
    n_matched
)

print(
    "Unmatched:",
    n_unmatched
)

print(
    "Spatial match rate:",
    round(
        match_rate,
        2
    ),
    "%"
)


# 7. Compare old CAS ID with spatially assigned 2023 ID

comparison = (
    crash_mb.loc[
        crash_mb[
            "MB2023_V1_00"
        ].notna()
    ]
    .copy()
)

comparison[
    "old_meshblock_key"
] = (
    pd.to_numeric(
        comparison[
            "meshblockId"
        ],
        errors="coerce"
    )
    .astype("Int64")
)

comparison[
    "spatial_2023_key"
] = (
    pd.to_numeric(
        comparison[
            "MB2023_V1_00"
        ],
        errors="coerce"
    )
    .astype("Int64")
)

both_available = (
    comparison[
        "old_meshblock_key"
    ].notna()
    &
    comparison[
        "spatial_2023_key"
    ].notna()
)

same_id = (
    comparison.loc[
        both_available,
        "old_meshblock_key"
    ]
    ==
    comparison.loc[
        both_available,
        "spatial_2023_key"
    ]
)


print(
    "\nCrashes with both old and spatial IDs:",
    both_available.sum()
)

print(
    "Old CAS meshblock ID equals spatial 2023 ID:",
    same_id.sum()
)

print(
    "Agreement rate:",
    round(
        same_id.mean() * 100,
        2
    ),
    "%"
)

Total CAS crashes: 34866
Missing X coordinates: 0
Missing Y coordinates: 0

Crash points created: 34866
Crash CRS: EPSG:2193

Usable meshblock polygons: 57523

Crash points matched to multiple meshblocks: 0

Spatial Match Results
Crash points: 34866
Matched to 2023 meshblock: 34866
Unmatched: 0
Spatial match rate: 100.0 %

Crashes with both old and spatial IDs: 34728
Old CAS meshblock ID equals spatial 2023 ID: 23566
Agreement rate: 67.86 %


In [4]:
# Join NZDep2023 to spatially assigned 2023 meshblocks

import pandas as pd


# 1. Prepare NZDep2023 join key

nzdep_join = nzdep.copy()

nzdep_join[
    "meshblock_2023_key"
] = (
    pd.to_numeric(
        nzdep_join[
            "MB2023_code"
        ],
        errors="coerce"
    )
    .astype("Int64")
)


# Spatially assigned meshblock key

crash_mb[
    "meshblock_2023_key"
] = (
    pd.to_numeric(
        crash_mb[
            "MB2023_V1_00"
        ],
        errors="coerce"
    )
    .astype("Int64")
)


# 2. Validate NZDep key uniqueness

print(
    "NZDep2023 rows:",
    len(nzdep_join)
)

print(
    "Unique NZDep meshblock keys:",
    nzdep_join[
        "meshblock_2023_key"
    ].nunique()
)

print(
    "Duplicate NZDep meshblock keys:",
    nzdep_join[
        "meshblock_2023_key"
    ]
    .duplicated()
    .sum()
)


# 3. Merge NZDep onto crash records

crash_nzdep = crash_mb.merge(
    nzdep_join[
        [
            "meshblock_2023_key",
            "NZDep2023",
            "NZDep2023_Score",
            "SA12023_code"
        ]
    ],
    on="meshblock_2023_key",
    how="left",
    validate="many_to_one"
)


# 4. Validate row count

print(
    "\nMerge Validation"
)

print(
    "Rows before NZDep merge:",
    len(crash_mb)
)

print(
    "Rows after NZDep merge:",
    len(crash_nzdep)
)


# 5. NZDep coverage

nzdep_available = (
    crash_nzdep[
        "NZDep2023"
    ]
    .notna()
)

print(
    "\nCrashes with NZDep2023:",
    nzdep_available.sum()
)

print(
    "Crashes without NZDep2023:",
    (~nzdep_available).sum()
)

print(
    "NZDep2023 coverage:",
    round(
        nzdep_available.mean() * 100,
        2
    ),
    "%"
)


# 6. Inspect deprivation decile distribution

print(
    "\nNZDep2023 decile distribution:"
)

print(
    crash_nzdep[
        "NZDep2023"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
)


# 7. Inspect NZDep score

print(
    "\nNZDep2023 score summary:"
)

print(
    crash_nzdep[
        "NZDep2023_Score"
    ]
    .describe()
)


# 8. NZDep availability by crash severity

print(
    "\nNZDep availability by crash severity:"
)

coverage_by_severity = pd.crosstab(
    crash_nzdep[
        "crashSeverity"
    ],
    crash_nzdep[
        "NZDep2023"
    ].notna(),
    margins=True
)

print(
    coverage_by_severity
)

NZDep2023 rows: 56382
Unique NZDep meshblock keys: 56382
Duplicate NZDep meshblock keys: 0

Merge Validation
Rows before NZDep merge: 34866
Rows after NZDep merge: 34866

Crashes with NZDep2023: 33887
Crashes without NZDep2023: 979
NZDep2023 coverage: 97.19 %

NZDep2023 decile distribution:
NZDep2023
1.0     1466
2.0     2478
3.0     2721
4.0     3070
5.0     3310
6.0     4025
7.0     3917
8.0     4324
9.0     4361
10.0    4215
NaN      979
Name: count, dtype: int64

NZDep2023 score summary:
count    33887.000000
mean      1019.898722
std         97.450001
min        827.000000
25%        949.000000
50%       1000.000000
75%       1072.000000
max       1713.000000
Name: NZDep2023_Score, dtype: float64

NZDep availability by crash severity:
NZDep2023         False   True    All
crashSeverity                        
Fatal Crash           8    319    327
Minor Crash         279   9718   9997
Non-Injury Crash    623  21413  22036
Serious Crash        69   2437   2506
All                 97

In [5]:
# NZDep2023 exploratory validation before modelling

import pandas as pd
from scipy.stats import spearmanr, kruskal


# 1. Missingness percentage by severity

missing_by_severity = (
    crash_nzdep
    .groupby("crashSeverity")
    .agg(
        Total=("crashSeverity", "size"),
        NZDep_Missing=(
            "NZDep2023",
            lambda x: x.isna().sum()
        )
    )
)

missing_by_severity["Missing %"] = (
    missing_by_severity["NZDep_Missing"]
    / missing_by_severity["Total"]
    * 100
).round(2)

print(
    "NZDep Missingness by Severity"
)

display(
    missing_by_severity
)


# 2. Relationship between NZDep decile and score

valid_dep = (
    crash_nzdep[
        crash_nzdep["NZDep2023"].notna()
        & crash_nzdep["NZDep2023_Score"].notna()
    ]
    .copy()
)

rho, rho_p = spearmanr(
    valid_dep["NZDep2023"],
    valid_dep["NZDep2023_Score"]
)

print(
    "\nDecile vs Score Relationship"
)

print(
    "Spearman correlation:",
    round(rho, 4)
)

print(
    "p-value:",
    rho_p
)


# 3. NZDep distributions by crash severity

severity_dep_summary = (
    crash_nzdep
    .groupby("crashSeverity")
    .agg(
        Count=(
            "NZDep2023",
            "count"
        ),
        Decile_Mean=(
            "NZDep2023",
            "mean"
        ),
        Decile_Median=(
            "NZDep2023",
            "median"
        ),
        Score_Mean=(
            "NZDep2023_Score",
            "mean"
        ),
        Score_Median=(
            "NZDep2023_Score",
            "median"
        )
    )
    .round(2)
)

print(
    "\nNZDep by Crash Severity"
)

display(
    severity_dep_summary
)


# 4. Statistical test across severity groups
# Kruskal-Wallis is suitable because NZDep decile is ordinal.

groups = [
    group["NZDep2023"]
    .dropna()
    .values
    for _, group
    in crash_nzdep.groupby(
        "crashSeverity"
    )
]

H, p = kruskal(
    *groups
)

n = sum(
    len(g)
    for g in groups
)

k = len(
    groups
)

epsilon_squared = max(
    0,
    (H - k + 1)
    / (n - k)
)

print(
    "\nKruskal-Wallis Test: NZDep Decile by Severity"
)

print(
    "H statistic:",
    round(H, 4)
)

print(
    "p-value:",
    p
)

print(
    "Epsilon-squared:",
    round(
        epsilon_squared,
        4
    )
)

NZDep Missingness by Severity


,Total,NZDep_Missing,Missing %
crashSeverity,,,
Fatal Crash,327,8,2.45
Minor Crash,9997,279,2.79
Non-Injury Crash,22036,623,2.83
Serious Crash,2506,69,2.75



Decile vs Score Relationship
Spearman correlation: 0.994
p-value: 0.0

NZDep by Crash Severity


,Count,Decile_Mean,Decile_Median,Score_Mean,Score_Median
crashSeverity,,,,,
Fatal Crash,319,5.56,5.0,998.29,976.0
Minor Crash,9718,6.13,6.0,1016.63,997.0
Non-Injury Crash,21413,6.29,7.0,1021.97,1003.0
Serious Crash,2437,6.14,6.0,1017.58,997.0



Kruskal-Wallis Test: NZDep Decile by Severity
H statistic: 48.16
p-value: 1.9689309981994533e-10
Epsilon-squared: 0.0013


### NZDep2023 Exploratory Analysis

NZDep2023 coverage was high, with deprivation information available for
97.19% of the 34,866 recorded crashes. Missingness was similar across all
four crash-severity classes, ranging from 2.45% to 2.83%.

NZDep2023 decile and NZDep2023 score were very strongly correlated
(Spearman's rho = 0.994), indicating that they represent largely the same
underlying socioeconomic dimension. They were therefore evaluated
separately rather than being included together in the initial models.

The distribution of NZDep deciles differed statistically across crash
severity classes (Kruskal-Wallis H = 48.16, p < 0.001). However, the
effect size was very small (epsilon-squared = 0.0013). Therefore, although
a statistical difference was detectable in the large dataset, the
univariate association between area-level deprivation and crash severity
was weak.

NZDep2023 is an area-level contextual measure and the observed associations
should not be interpreted as causal effects at either the crash or
individual level.

In [6]:
# Prepare datasets for NZDep feature-ablation experiment

import pandas as pd
import numpy as np


# 1. Baseline CAS predictors
# Same 30 predictors used in the national baseline

baseline_features = [
    "bicycle",
    "bus",
    "carStationWagon",
    "crashDirectionDescription",
    "crashFinancialYear",
    "crashSHDescription",
    "crashYear",
    "directionRoleDescription",
    "flatHill",
    "light",
    "moped",
    "motorcycle",
    "NumberOfLanes",
    "otherVehicleType",
    "region",
    "roadCharacter",
    "roadLane",
    "roadSurface",
    "schoolBus",
    "speedLimit",
    "streetLight",
    "suv",
    "taxi",
    "trafficControl",
    "truck",
    "unknownVehicleType",
    "urban",
    "vanOrUtility",
    "weatherA",
    "weatherB"
]

target = "crashSeverity"


# 2. Create modelling dataframe

nzdep_model_df = crash_nzdep[
    baseline_features
    + [
        "NZDep2023",
        "NZDep2023_Score",
        target
    ]
].copy()


# 3. Missing-value indicator
# 1 = crash has no available NZDep value
# 0 = NZDep is available

nzdep_model_df["NZDep_missing"] = (
    nzdep_model_df["NZDep2023"]
    .isna()
    .astype(int)
)


# 4. Define feature sets for ablation

feature_sets = {

    "CAS Only":
        baseline_features,

    "CAS + NZDep Decile":
        baseline_features
        + ["NZDep2023"],

    "CAS + NZDep Score":
        baseline_features
        + ["NZDep2023_Score"],

    "CAS + NZDep Decile + Missing Indicator":
        baseline_features
        + [
            "NZDep2023",
            "NZDep_missing"
        ]
}


# 5. Validation

print(
    "Modelling dataframe shape:",
    nzdep_model_df.shape
)

print(
    "\nTotal crashes:",
    len(nzdep_model_df)
)

print(
    "Target missing:",
    nzdep_model_df[target]
    .isna()
    .sum()
)

print(
    "\nNZDep decile missing:",
    nzdep_model_df["NZDep2023"]
    .isna()
    .sum()
)

print(
    "NZDep score missing:",
    nzdep_model_df["NZDep2023_Score"]
    .isna()
    .sum()
)

print(
    "Missing-indicator count:",
    nzdep_model_df["NZDep_missing"]
    .sum()
)


print(
    "\nFeature Sets"
)

for name, features in feature_sets.items():

    print(
        f"{name}: {len(features)} predictors"
    )


# 6. Confirm all experiments retain the same population

print(
    "\nPopulation Check"
)

for name in feature_sets:

    print(
        f"{name}: "
        f"{len(nzdep_model_df):,} crashes"
    )


# 7. Target distribution

print(
    "\nTarget Distribution"
)

print(
    nzdep_model_df[target]
    .value_counts()
)

print(
    "\nTarget percentages:"
)

print(
    (
        nzdep_model_df[target]
        .value_counts(
            normalize=True
        )
        * 100
    ).round(2)
)

Modelling dataframe shape: (34866, 34)

Total crashes: 34866
Target missing: 0

NZDep decile missing: 979
NZDep score missing: 979
Missing-indicator count: 979

Feature Sets
CAS Only: 30 predictors
CAS + NZDep Decile: 31 predictors
CAS + NZDep Score: 31 predictors
CAS + NZDep Decile + Missing Indicator: 32 predictors

Population Check
CAS Only: 34,866 crashes
CAS + NZDep Decile: 34,866 crashes
CAS + NZDep Score: 34,866 crashes
CAS + NZDep Decile + Missing Indicator: 34,866 crashes

Target Distribution
crashSeverity
Non-Injury Crash    22036
Minor Crash          9997
Serious Crash        2506
Fatal Crash           327
Name: count, dtype: int64

Target percentages:
crashSeverity
Non-Injury Crash    63.20
Minor Crash         28.67
Serious Crash        7.19
Fatal Crash          0.94
Name: proportion, dtype: float64


In [7]:
# NZDep2023 Feature Ablation
# Weighted XGBoost
# Same national CAS population and seed-42 split

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    LabelEncoder
)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score
)

from xgboost import XGBClassifier


# 1. Original categorical CAS predictors

baseline_categorical = [
    "crashDirectionDescription",
    "crashFinancialYear",
    "crashSHDescription",
    "directionRoleDescription",
    "flatHill",
    "light",
    "region",
    "roadCharacter",
    "roadLane",
    "roadSurface",
    "streetLight",
    "trafficControl",
    "urban",
    "weatherA",
    "weatherB"
]


# 2. Create one fixed train/test split
# Every feature set uses exactly the same
# training and test crashes.

all_indices = nzdep_model_df.index

train_idx, test_idx = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=42,
    stratify=nzdep_model_df.loc[
        all_indices,
        target
    ]
)


y_train = nzdep_model_df.loc[
    train_idx,
    target
].copy()

y_test = nzdep_model_df.loc[
    test_idx,
    target
].copy()


print(
    "Training crashes:",
    len(train_idx)
)

print(
    "Test crashes:",
    len(test_idx)
)


# 3. Encode target once

label_encoder = LabelEncoder()

y_train_encoded = (
    label_encoder.fit_transform(
        y_train
    )
)

y_test_encoded = (
    label_encoder.transform(
        y_test
    )
)


print(
    "\nClass mapping:"
)

for i, label in enumerate(
    label_encoder.classes_
):
    print(
        i,
        "=",
        label
    )


fatal_id = list(
    label_encoder.classes_
).index(
    "Fatal Crash"
)

serious_id = list(
    label_encoder.classes_
).index(
    "Serious Crash"
)


# 4. Balanced sample weights

sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train_encoded
)


# 5. Evaluation function

def evaluate_nzdep_feature_set(
    experiment_name,
    feature_list
):

    # Same records for every experiment

    X_train = nzdep_model_df.loc[
        train_idx,
        feature_list
    ].copy()

    X_test = nzdep_model_df.loc[
        test_idx,
        feature_list
    ].copy()


    # Identify categorical and numeric predictors

    categorical_list = [
        col
        for col in feature_list
        if col in baseline_categorical
    ]

    numeric_list = [
        col
        for col in feature_list
        if col not in categorical_list
    ]


    # Preprocessing
    # Fit only on training data

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])


    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ])


    preprocessor = ColumnTransformer([
        (
            "num",
            numeric_pipeline,
            numeric_list
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_list
        )
    ])


    X_train_processed = (
        preprocessor.fit_transform(
            X_train
        )
    )

    X_test_processed = (
        preprocessor.transform(
            X_test
        )
    )


    # Weighted XGBoost configuration

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    )


    model.fit(
        X_train_processed,
        y_train_encoded,
        sample_weight=sample_weights
    )


    predictions = model.predict(
        X_test_processed
    )


    # Metrics

    fatal_recall = recall_score(
        y_test_encoded,
        predictions,
        labels=[fatal_id],
        average=None,
        zero_division=0
    )[0]

    serious_recall = recall_score(
        y_test_encoded,
        predictions,
        labels=[serious_id],
        average=None,
        zero_division=0
    )[0]


    return {
        "Feature Set":
            experiment_name,

        "Predictors":
            len(feature_list),

        "Accuracy":
            accuracy_score(
                y_test_encoded,
                predictions
            ),

        "Balanced Accuracy":
            balanced_accuracy_score(
                y_test_encoded,
                predictions
            ),

        "Macro F1":
            f1_score(
                y_test_encoded,
                predictions,
                average="macro"
            ),

        "Fatal Recall":
            fatal_recall,

        "Serious Recall":
            serious_recall
    }


# 6. Run ablation experiments

nzdep_results = []

for experiment_name, features in (
    feature_sets.items()
):

    print(
        "\nRunning:",
        experiment_name
    )

    result = evaluate_nzdep_feature_set(
        experiment_name,
        features
    )

    nzdep_results.append(
        result
    )


# 7. Results table

nzdep_ablation = pd.DataFrame(
    nzdep_results
)


# Calculate change relative to CAS-only

cas_macro_f1 = nzdep_ablation.loc[
    nzdep_ablation["Feature Set"]
    == "CAS Only",
    "Macro F1"
].iloc[0]


nzdep_ablation[
    "Macro F1 Change"
] = (
    nzdep_ablation[
        "Macro F1"
    ]
    - cas_macro_f1
)


nzdep_ablation = (
    nzdep_ablation
    .sort_values(
        "Macro F1",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    nzdep_ablation.round(4)
)

Training crashes: 27892
Test crashes: 6974

Class mapping:
0 = Fatal Crash
1 = Minor Crash
2 = Non-Injury Crash
3 = Serious Crash

Running: CAS Only

Running: CAS + NZDep Decile

Running: CAS + NZDep Score

Running: CAS + NZDep Decile + Missing Indicator


,Feature Set,Predictors,Accuracy,Balanced Accuracy,Macro F1,Fatal Recall,Serious Recall,Macro F1 Change
0,CAS + NZDep Score,31,0.4981,0.4107,0.3338,0.2923,0.4112,0.0066
1,CAS + NZDep Decile,31,0.4957,0.4011,0.3308,0.2615,0.4012,0.0035
2,CAS + NZDep Decile + Missing Indicator,32,0.4984,0.3954,0.3297,0.2462,0.3932,0.0025
3,CAS Only,30,0.4900,0.4076,0.3272,0.3231,0.3892,0.0000


### NZDep2023 Feature-Ablation Result

At random seed 42, the continuous NZDep2023 score produced the strongest
performance among the tested socioeconomic representations.

Adding NZDep2023 Score increased Macro F1 from 0.3272 for the CAS-only
baseline to 0.3338, an improvement of 0.0066. Balanced Accuracy also
increased slightly from 0.4076 to 0.4107, while overall Accuracy increased
from 0.4900 to 0.4981.

Serious-crash recall improved from 0.3892 to 0.4112. However, Fatal-crash
recall decreased from 0.3231 to 0.2923. Therefore, the socioeconomic feature
did not improve every evaluation measure.

NZDep2023 Score outperformed the deprivation decile and the
decile-plus-missing-indicator configuration in this split. This suggests
that the continuous deprivation score may preserve more useful predictive
information than the coarser decile representation.

Because these results are based on a single train/test split, multi-seed
evaluation is required before concluding that the improvement is robust.

In [8]:
# NZDep2023 Multi-Seed Robustness Experiment
# Seeds: 42, 7, 21, 84, 123

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    LabelEncoder
)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score
)

from xgboost import XGBClassifier


# 1. Seeds

seeds = [
    42,
    7,
    21,
    84,
    123
]


# 2. Evaluation function

def evaluate_nzdep_seed(
    experiment_name,
    feature_list,
    seed
):

    # Same population, new stratified split per seed

    indices = nzdep_model_df.index

    train_idx, test_idx = train_test_split(
        indices,
        test_size=0.20,
        random_state=seed,
        stratify=nzdep_model_df.loc[
            indices,
            target
        ]
    )


    X_train = nzdep_model_df.loc[
        train_idx,
        feature_list
    ].copy()

    X_test = nzdep_model_df.loc[
        test_idx,
        feature_list
    ].copy()

    y_train = nzdep_model_df.loc[
        train_idx,
        target
    ].copy()

    y_test = nzdep_model_df.loc[
        test_idx,
        target
    ].copy()


    # Categorical and numeric predictors

    categorical_list = [
        col
        for col in feature_list
        if col in baseline_categorical
    ]

    numeric_list = [
        col
        for col in feature_list
        if col not in categorical_list
    ]


    # Preprocessing

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])


    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ])


    preprocessor = ColumnTransformer([
        (
            "num",
            numeric_pipeline,
            numeric_list
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_list
        )
    ])


    X_train_processed = (
        preprocessor.fit_transform(
            X_train
        )
    )

    X_test_processed = (
        preprocessor.transform(
            X_test
        )
    )


    # Target encoding

    encoder = LabelEncoder()

    y_train_encoded = (
        encoder.fit_transform(
            y_train
        )
    )

    y_test_encoded = (
        encoder.transform(
            y_test
        )
    )


    fatal_id = list(
        encoder.classes_
    ).index(
        "Fatal Crash"
    )

    serious_id = list(
        encoder.classes_
    ).index(
        "Serious Crash"
    )


    # Balanced sample weights

    sample_weights = (
        compute_sample_weight(
            class_weight="balanced",
            y=y_train_encoded
        )
    )


    # Weighted XGBoost

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=seed,
        n_jobs=-1
    )


    model.fit(
        X_train_processed,
        y_train_encoded,
        sample_weight=sample_weights
    )


    predictions = model.predict(
        X_test_processed
    )


    # Metrics

    fatal_recall = recall_score(
        y_test_encoded,
        predictions,
        labels=[fatal_id],
        average=None,
        zero_division=0
    )[0]

    serious_recall = recall_score(
        y_test_encoded,
        predictions,
        labels=[serious_id],
        average=None,
        zero_division=0
    )[0]


    return {
        "Feature Set":
            experiment_name,

        "Seed":
            seed,

        "Accuracy":
            accuracy_score(
                y_test_encoded,
                predictions
            ),

        "Balanced Accuracy":
            balanced_accuracy_score(
                y_test_encoded,
                predictions
            ),

        "Macro F1":
            f1_score(
                y_test_encoded,
                predictions,
                average="macro"
            ),

        "Fatal Recall":
            fatal_recall,

        "Serious Recall":
            serious_recall
    }


# 3. Run all feature sets across all seeds

robustness_results = []

for seed in seeds:

    print(
        f"\nSeed {seed}"
    )

    for experiment_name, features in (
        feature_sets.items()
    ):

        print(
            "Running:",
            experiment_name
        )

        result = evaluate_nzdep_seed(
            experiment_name,
            features,
            seed
        )

        robustness_results.append(
            result
        )


nzdep_multiseed_raw = pd.DataFrame(
    robustness_results
)


# 4. Raw per-seed results

print(
    "\nPer-Seed Results"
)

display(
    nzdep_multiseed_raw.round(4)
)


# 5. Summary means and standard deviations

nzdep_multiseed_summary = (
    nzdep_multiseed_raw
    .groupby(
        "Feature Set"
    )
    .agg(
        Accuracy_Mean=(
            "Accuracy",
            "mean"
        ),
        Accuracy_SD=(
            "Accuracy",
            "std"
        ),
        Balanced_Accuracy_Mean=(
            "Balanced Accuracy",
            "mean"
        ),
        Balanced_Accuracy_SD=(
            "Balanced Accuracy",
            "std"
        ),
        Macro_F1_Mean=(
            "Macro F1",
            "mean"
        ),
        Macro_F1_SD=(
            "Macro F1",
            "std"
        ),
        Fatal_Recall_Mean=(
            "Fatal Recall",
            "mean"
        ),
        Fatal_Recall_SD=(
            "Fatal Recall",
            "std"
        ),
        Serious_Recall_Mean=(
            "Serious Recall",
            "mean"
        ),
        Serious_Recall_SD=(
            "Serious Recall",
            "std"
        )
    )
    .reset_index()
)


# 6. Mean Macro F1 change vs CAS baseline

cas_mean_macro = (
    nzdep_multiseed_summary.loc[
        nzdep_multiseed_summary[
            "Feature Set"
        ] == "CAS Only",
        "Macro_F1_Mean"
    ]
    .iloc[0]
)


nzdep_multiseed_summary[
    "Macro_F1_Change"
] = (
    nzdep_multiseed_summary[
        "Macro_F1_Mean"
    ]
    - cas_mean_macro
)


# 7. Count seeds where each feature set beats CAS Only

cas_seed_scores = (
    nzdep_multiseed_raw[
        nzdep_multiseed_raw[
            "Feature Set"
        ] == "CAS Only"
    ]
    .set_index("Seed")[
        "Macro F1"
    ]
)


improved_seed_counts = {}

for experiment_name in feature_sets:

    if experiment_name == "CAS Only":

        improved_seed_counts[
            experiment_name
        ] = "-"

    else:

        exp_scores = (
            nzdep_multiseed_raw[
                nzdep_multiseed_raw[
                    "Feature Set"
                ] == experiment_name
            ]
            .set_index("Seed")[
                "Macro F1"
            ]
        )

        improved = (
            exp_scores
            > cas_seed_scores
        ).sum()

        improved_seed_counts[
            experiment_name
        ] = f"{improved}/{len(seeds)}"


nzdep_multiseed_summary[
    "Seeds Improved"
] = (
    nzdep_multiseed_summary[
        "Feature Set"
    ]
    .map(
        improved_seed_counts
    )
)


# 8. Sort by mean Macro F1

nzdep_multiseed_summary = (
    nzdep_multiseed_summary
    .sort_values(
        "Macro_F1_Mean",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nFive-Seed Summary"
)

display(
    nzdep_multiseed_summary
    .round(4)
)


Seed 42
Running: CAS Only
Running: CAS + NZDep Decile
Running: CAS + NZDep Score
Running: CAS + NZDep Decile + Missing Indicator

Seed 7
Running: CAS Only
Running: CAS + NZDep Decile
Running: CAS + NZDep Score
Running: CAS + NZDep Decile + Missing Indicator

Seed 21
Running: CAS Only
Running: CAS + NZDep Decile
Running: CAS + NZDep Score
Running: CAS + NZDep Decile + Missing Indicator

Seed 84
Running: CAS Only
Running: CAS + NZDep Decile
Running: CAS + NZDep Score
Running: CAS + NZDep Decile + Missing Indicator

Seed 123
Running: CAS Only
Running: CAS + NZDep Decile
Running: CAS + NZDep Score
Running: CAS + NZDep Decile + Missing Indicator

Per-Seed Results


,Feature Set,Seed,Accuracy,Balanced Accuracy,Macro F1,Fatal Recall,Serious Recall
0,CAS Only,42,0.4900,0.4076,0.3272,0.3231,0.3892
1,CAS + NZDep Decile,42,0.4957,0.4011,0.3308,0.2615,0.4012
2,CAS + NZDep Score,42,0.4981,0.4107,0.3338,0.2923,0.4112
3,CAS + NZDep Decile + Missing Indicator,42,0.4984,0.3954,0.3297,0.2462,0.3932
4,CAS Only,7,0.4920,0.4111,0.3284,0.3385,0.3892
5,CAS + NZDep Decile,7,0.4943,0.3987,0.3258,0.2923,0.3832
6,CAS + NZDep Score,7,0.5020,0.3982,0.3308,0.2615,0.3872
7,CAS + NZDep Decile + Missing Indicator,7,0.4967,0.4036,0.3277,0.3077,0.3812
8,CAS Only,21,0.4991,0.4217,0.3351,0.3538,0.4012
9,CAS + NZDep Decile,21,0.4996,0.3996,0.3292,0.2769,0.3932



Five-Seed Summary


,Feature Set,Accuracy_Mean,Accuracy_SD,Balanced_Accuracy_Mean,Balanced_Accuracy_SD,Macro_F1_Mean,Macro_F1_SD,Fatal_Recall_Mean,Fatal_Recall_SD,Serious_Recall_Mean,Serious_Recall_SD,Macro_F1_Change,Seeds Improved
0,CAS + NZDep Score,0.5023,0.0035,0.3989,0.0103,0.3314,0.0016,0.2585,0.0515,0.3940,0.0141,0.0026,3/5
1,CAS + NZDep Decile,0.4985,0.0035,0.4014,0.0046,0.3302,0.0033,0.2738,0.0129,0.3988,0.0141,0.0014,3/5
2,CAS + NZDep Decile + Missing Indicator,0.4991,0.0036,0.3972,0.0041,0.3294,0.0025,0.2615,0.0344,0.3928,0.0150,0.0006,3/5
3,CAS Only,0.4940,0.0039,0.4062,0.0123,0.3288,0.0049,0.3077,0.0435,0.3944,0.0157,0.0000,-


### Robustness of the NZDep2023 Contribution

The five-seed robustness experiment showed that the predictive contribution
of area-level socioeconomic deprivation was modest.

CAS + NZDep2023 Score achieved the highest mean Macro F1 (0.3314), compared
with 0.3288 for the CAS-only baseline, representing an average improvement
of 0.0026. The NZDep Score configuration outperformed the CAS-only baseline
in three of five random-seed experiments.

However, the improvement was not consistent across all evaluation measures.
Mean Balanced Accuracy decreased from 0.4062 for CAS-only to 0.3989 with
NZDep Score. Mean Fatal-crash recall also decreased from 0.3077 to 0.2585,
while Serious-crash recall remained almost unchanged.

The NZDep decile and decile-plus-missing-indicator configurations produced
even smaller mean Macro F1 improvements.

Therefore, NZDep2023 provides a small amount of additional predictive
information, but the effect is not sufficiently strong or consistent to
claim a substantial improvement in crash-severity classification. The
continuous NZDep2023 Score was retained as the preferred socioeconomic
representation for subsequent cross-dataset experiments because it achieved
the highest mean Macro F1 among the tested NZDep configurations.

In [9]:
# Save validated NZDep2023 crash-level enrichment

NZDEP_ENRICHED_PATH = (
    PROJECT_ROOT
    / "data"
    / "nzdep2023"
    / "CAS_NZDep2023_enriched.csv"
)

nzdep_export = crash_nzdep[
    [
        "cas_row_id",
        "X",
        "Y",
        "crashSeverity",
        "MB2023_V1_00",
        "NZDep2023",
        "NZDep2023_Score",
        "SA12023_code"
    ]
].copy()

nzdep_export.to_csv(
    NZDEP_ENRICHED_PATH,
    index=False
)

print(
    "Saved:"
)

print(
    NZDEP_ENRICHED_PATH
)

print(
    "\nShape:",
    nzdep_export.shape
)

print(
    "\nMissing NZDep Score:"
)

print(
    nzdep_export[
        "NZDep2023_Score"
    ]
    .isna()
    .sum()
)

print(
    "\nDuplicate cas_row_id:"
)

print(
    nzdep_export[
        "cas_row_id"
    ]
    .duplicated()
    .sum()
)

Saved:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\nzdep2023\CAS_NZDep2023_enriched.csv

Shape: (34866, 8)

Missing NZDep Score:
979

Duplicate cas_row_id:
0
